# 📝 OpenAI API 활용 과제 LV3(통합) — 리뷰 리포트 · 상담 어시스턴트

> 지금까지 배운 것을 묶어 **작은 프로그램 두 개**를 완성합니다. 각 프로그램은 **여러 단계**로 나뉘어 있고, **각 단계 셀**에 그 단계에서 만들 변수와 조건이 적혀 있습니다.

## 풀이 방법
1. 맨 위 **준비 셀**들을 먼저 실행하세요(`.env` 에 본인 API 키가 있어야 합니다).
2. 각 단계의 **답안 셀**을 채우고 아래 **자가채점 셀**로 확인하세요.
3. 단계는 **순서대로** 풀어야 합니다(앞 단계 변수를 뒤 단계에서 씁니다).

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 클라이언트 준비 — 이 셀을 먼저 실행하세요.
# .env 파일에 OPENAI_API_KEY 를 넣어 두면 아래 한 줄이 그것을 읽어 연결합니다.
#   참고: https://developers.openai.com/api/docs/guides/text
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv('.env')                # 같은 폴더의 .env
load_dotenv('../.env')             # (정답 폴더처럼 한 단계 안에서 열었을 때)

# max_retries: 분당 토큰 한도(TPM)에 걸리면(429) 잠시 뒤 자동으로 다시 시도한다.
#   이미지는 한 장에 수만 토큰이라 여러 장을 연달아 보내면 쉽게 걸린다.
client = OpenAI(max_retries=8)     # OPENAI_API_KEY 를 자동으로 찾아 쓴다
print('연결 준비 완료 —', '키 확인됨' if os.getenv('OPENAI_API_KEY') else '키가 없습니다(.env 를 확인하세요)')

In [ ]:
# [제공 코드] 라이브러리
import pandas as pd


In [ ]:
# [제공 코드] 감정분석 스키마 + analyze() (교안에서 만든 그대로 — 그냥 실행하세요)
import json
senti_schema = {'type': 'json_schema', 'json_schema': {
    'name': 'review_sentiment',
    'schema': {'type': 'object',
        'properties': {
            'sentiment': {'type': 'string', 'enum': ['긍정', '부정', '중립']},
            'summary': {'type': 'string'}},
        'required': ['sentiment', 'summary'], 'additionalProperties': False},
    'strict': True}}

def analyze(text):
    resp = client.chat.completions.create(model='gpt-4o-mini',
        messages=[{'role': 'system', 'content': '리뷰 감정을 분석해 스키마에 맞춰 답해.'},
                  {'role': 'user', 'content': str(text)}],
        response_format=senti_schema, temperature=0)
    return json.loads(resp.choices[0].message.content)

---
# 프로그램 ① 자세밴드 리뷰 감정분석 리포트

리뷰 데이터를 받아 **감정을 분석 → 집계 → 경영진용 리포트**까지 자동 생성하는 프로그램을 단계별로 완성합니다. 데이터는 `data/reviews.csv`(자세밴드 40건, 열: `review_id`·`product`·`rating`·`content`).

### 1단계 — 데이터 로드·살펴보기
`data/reviews.csv` 를 읽어 변수 **`reviews`** 에 담고, `shape`·`head()`·별점 분포를 확인하세요. (살펴보기 출력은 자유. 채점은 `reviews` 가 DataFrame 이고 40행인지만 봅니다.)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(reviews, pd.DataFrame) and reviews.shape[0] == 40
print('✅ 통과!')

### 2단계 — 배치 감정분석
제공된 `analyze()` 로 리뷰를 분석해 결과 딕셔너리 리스트 **`results`** 에 담으세요. (각 결과는 `sentiment`·`summary` 키를 가집니다.)

⚠️ **앞에서부터 자르지 마세요.** 이 CSV 는 **별점 오름차순으로 정렬**돼 있어 `head(12)` 로 자르면 1~2점 리뷰만 뽑혀 **감정 분포가 한쪽으로 쏠립니다**(3단계 집계가 의미를 잃습니다). **별점별로 고르게** 3건씩 뽑아 총 **15건**을 분석하세요 — `sample` 변수에 담고 그것을 도세요.

⚠️ 그리고 **한 건이 실패해도 멈추지 않게** 만드세요(교안 3교시 5절). 실제 데이터에는 빈 칸이 섞여 들어옵니다 — 그 상황을 직접 만들어 확인합니다.

- `sample['content']` 를 리스트로 만들고 **여덟 번째 항목(`[7]`)을 빈 문자열로 바꿔** 변수 **`texts_with_hole`** 에 담으세요(실제 데이터의 빈 칸을 흉내 낸 것입니다).
- 그 리스트를 돌며 분석하되, **내용이 없으면 호출하지 말고** 나머지는 `try/except` 로 감싸세요. 빈 자리·실패한 자리에는 **`None` 을 대신 담아** `results` 의 길이가 15로 유지되게 하세요.

**예시**
```
sample = reviews.groupby('rating', group_keys=False).head(3)   # 별점 1~5 × 3건 = 15건
len(results)  →  15
```
<details><summary>힌트</summary>

```text
세부구현:
1. groupby('rating') 후 각 그룹에서 head(3) 을 뽑아 sample 에 담는다(group_keys=False).
2. list(sample['content']) 로 만든 뒤 [7] 자리를 '' 로 바꾼다(변수 texts_with_hole).
3. 빈 리스트로 시작해(변수 results) texts_with_hole 을 돈다.
4. 내용이 없으면 None 을 담고 continue, 아니면 try/except 로 analyze 를 부른다(실패해도 None).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(results) == 15, 'results 길이는 sample 과 같아야 합니다(실패한 자리는 None 으로 남깁니다)'
assert results[7] is None, '빈 칸 자리는 None 이어야 합니다 — 한 건 실패가 전체를 멈추면 안 됩니다'
assert all(set(r.keys()) == {'sentiment', 'summary'} for r in results if r)
assert sum(1 for r in results if r) == 14, '나머지 14건은 정상 분석돼야 합니다'
# 별점별로 고르게 뽑았는지 — 한쪽 별점만 담겼으면 분포 집계가 무의미하다
assert dict(sample['rating'].value_counts().sort_index()) == {1: 3, 2: 3, 3: 3, 4: 3, 5: 3}
print('✅ 통과!')

### 3단계 — 집계
`results` 를 DataFrame 으로 만들어, 감정 분포를 센 딕셔너리 **`dist`**(예 `{'긍정':7,'부정':6,'중립':2}` — 합이 표본 15건)와 부정 리뷰들의 요약(`summary`) 리스트 **`neg_summaries`** 를 만드세요.

<details><summary>힌트</summary>

```text
세부구현:
1. results 로 DataFrame 을 만든다.
2. sentiment 열의 값별 개수를 세어 딕셔너리로 만든다(변수 dist).
3. sentiment 가 '부정'인 행의 summary 열만 리스트로 모은다(변수 neg_summaries).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(dist, dict) and sum(dist.values()) == 14, '건너뛴 한 건을 빼고 14건이어야 합니다'
assert isinstance(neg_summaries, list)
# 별점을 고르게 뽑았으니 감정도 한 종류로 몰리면 안 된다 — 분포다운 분포인지 확인
assert len(dist) >= 2, '감정이 한 종류뿐이면 표본이 한쪽으로 쏠린 것입니다'
assert len(neg_summaries) > 0, '저평점 리뷰를 포함했다면 부정 요약이 있어야 합니다'
print('✅ 통과!')

### 4단계 — LLM 리포트 생성
집계 결과(`dist`)와 부정 요약(`neg_summaries`)을 프롬프트에 넣어, `gpt-4o-mini` 로 **경영진용 3줄 요약 리포트**를 생성해 문자열 **`report`** 에 담으세요. (프롬프트에 감정 분포 숫자와 부정 요약을 포함하고, '경영진이 읽을 3줄 리포트로' 라고 지시하세요.)

<details><summary>힌트</summary>

```text
세부구현:
1. content 프롬프트에 f-string 으로 dist 와 '\n'.join(neg_summaries) 를 넣는다.
2. '위 결과를 경영진용 3줄 리포트로 정리해줘' 를 덧붙여 create 호출.
3. 답 content 를 report 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(report, str) and len(report.strip()) > 0
print('✅ 통과!')

### 5단계 — 리포트를 파일로 저장
완성한 리포트는 파일로 남겨야 쓸모가 있습니다. 3단계의 감정 분포(`dist`)와 4단계의 리포트(`report`)를 하나의 텍스트로 묶어 **`output/sentiment_report.txt`** 에 저장하세요(4일차에서 배운 파일 쓰기).

- `output` 폴더가 없으면 만드세요(`os.makedirs('output', exist_ok=True)`).
- 저장한 파일 경로 문자열을 변수 **`report_path`** 에 담으세요.
- 파일 안에는 **감정 분포(`dist`)와 리포트 본문(`report`)이 둘 다** 들어가야 합니다(꾸미는 형식은 자유).

<details><summary>힌트</summary>

```text
세부구현:
1. os.makedirs('output', exist_ok=True) 로 폴더를 준비한다.
2. report_path 에 'output/sentiment_report.txt' 를 담는다.
3. with open(report_path, 'w', encoding='utf-8') 로 dist 와 report 를 이어 쓴다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
assert os.path.exists(report_path)
saved = open(report_path, encoding='utf-8').read()
# 리포트 본문과 감정 분포가 둘 다 담겼는지 (꾸미는 형식은 자유)
assert report.strip() in saved, '4단계의 report 본문이 파일에 들어 있어야 합니다'
assert all(k in saved for k in dist), '감정 분포(dist)도 함께 저장해야 합니다'
print('✅ 통과!')

---
# 프로그램 ② 선크림 리뷰 상담 어시스턴트

사용자의 질문에 따라 **알맞은 도구(함수)를 골라 실행**하고 답하는 어시스턴트를 만듭니다. 데이터는 `data/reviews_sun.csv`(선크림 20건). Function Calling 으로 **여러 도구 + 라우팅**을 구현합니다.

### 1단계 — 데이터·도구 함수 준비
`data/reviews_sun.csv` 를 **`sun`** 에 담으세요. 도구로 쓸 함수 세 개는 아래 제공 셀에 있습니다(별점별 개수·키워드 개수·평균 별점). 실행만 하면 됩니다.

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert isinstance(sun, pd.DataFrame) and sun.shape[0] == 20
print('✅ 통과!')

In [ ]:
# [제공 코드] 도구로 쓸 함수 세 개
def count_by_rating(rating):
    return int((sun['rating'] == rating).sum())

def count_keyword(keyword):
    return int(sun['content'].str.contains(keyword).sum())

def avg_rating():
    return round(float(sun['rating'].mean()), 2)

### 2단계 — 도구 스키마와 디스패처
세 함수를 모델에게 알려 줄 `tools` 스키마 리스트 **`tools`** 를 만들고, 함수 이름을 실제 함수에 연결하는 **`dispatch`** 딕셔너리(`{'count_by_rating': count_by_rating, ...}`)를 만드세요.

- `count_by_rating`: 정수 인자 `rating`
- `count_keyword`: 문자열 인자 `keyword`
- `avg_rating`: 인자 없음(빈 properties)

<details><summary>힌트</summary>

```text
세부구현:
1. tools 리스트에 세 함수의 스키마를 각각 넣는다(avg_rating 은 properties 를 빈 딕셔너리로).
2. 함수 이름(문자열)을 키로, 실제 함수를 값으로 하는 딕셔너리를 만든다(변수 dispatch).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(tools) == 3
assert set(dispatch.keys()) == {'count_by_rating', 'count_keyword', 'avg_rating'}
print('✅ 통과!')

### 3단계 — 어시스턴트 함수
질문을 받아, 모델이 도구를 부르면 **디스패처로 실행 → 결과를 넣어 재호출 → 최종 답**을 하고, 도구가 필요 없으면 그냥 답하는 함수 **`ask_assistant(question)`** 를 만드세요(문자열을 반환).

<details><summary>힌트</summary>

```text
접근방법:
- 교안 2절의 4단계 흐름을 함수로 감싼다. tool_calls 유무로 분기한다.

세부구현:
1. 질문 하나로 messages 를 만들어 tools 를 넣어 1차 호출하고, 첫 메시지를 꺼낸다.
2. 그 메시지에 tool_calls 가 없으면 그대로 content 를 반환한다.
3. tool_calls 가 있으면 첫 호출의 함수 이름과 인자(json.loads)를 꺼내, dispatch 에서 그 이름의 함수를 찾아 인자로 실행한다.
4. assistant(tool_calls) 메시지와 tool(결과) 메시지를 붙여 2차 호출한 뒤 content 를 반환한다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
reply = ask_assistant('별점 5점 리뷰 몇 개야?')
assert isinstance(reply, str) and len(reply.strip()) > 0
print('✅ 통과!')

### 4단계 — 여러 질문으로 시험
아래 세 질문을 `ask_assistant` 로 처리해 답 문자열들을 리스트 **`answers`** 에 담으세요. 앞의 두 질문은 **도구가 필요하고**(평균 별점·키워드 개수), 마지막 질문은 도구 없이 모델이 바로 답합니다 — 세 답이 모두 나오면 라우팅이 제대로 도는 것입니다.

- 도구가 필요한 질문은 반드시 **`dispatch` 를 거쳐** 실제 함수가 실행돼야 합니다(자가채점이 이를 확인합니다).

```python
questions = ['평균 별점이 몇 점이야?', '백탁 언급한 리뷰 몇 개야?', '선크림 바를 때 팁 하나 알려줘.']
```

<details><summary>힌트</summary>

```text
세부구현:
1. 세 질문을 리스트로 만든다(변수 questions).
2. 각 질문을 ask_assistant 로 처리한 답들을 리스트로 모은다(변수 answers).
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(answers) == 3
assert all(isinstance(a, str) and len(a.strip()) > 0 for a in answers)

# 도구가 실제로 실행됐는지 — dispatch 의 함수를 잠깐 감싸 호출된 이름을 기록해 본다
called = []
real_fns = dict(dispatch)
def make_recorder(name, fn):
    def recorded(**kwargs):
        called.append(name)
        return fn(**kwargs)
    return recorded
for name, fn in real_fns.items():
    dispatch[name] = make_recorder(name, fn)
ask_assistant('평균 별점이 몇 점이야?')
dispatch.update(real_fns)          # 원래 함수로 되돌린다
assert 'avg_rating' in called, \
    '평균 별점 질문에는 avg_rating 도구가 dispatch 를 거쳐 실행돼야 합니다'
print('✅ 통과!')

---
# 프로그램 ③ 영수증 정산 파이프라인

영수증 **사진**을 읽어 품목·금액 표로 만들고, **스스로 검산**한 뒤 파일로 남기는 프로그램입니다. 데이터는 `data/receipt/` 의 영수증 사진과, 사람이 정리해 둔 정답표 `data/receipts.csv` 입니다. (4교시의 이미지 정형화를 처음부터 끝까지 직접 만듭니다.)

### 1단계 — 영수증 스키마 설계
영수증 한 장에서 받을 값을 pydantic 스키마로 **직접** 선언하세요.

- **`ReceiptItem`** : `name`(str) · `quantity`(int) · `price`(float, 그 줄의 합계 금액)
- **`Receipt`** : `store_name`(**Optional[str]**, 안 보이면 None) · `total_amount`(float) · `payment`(**Literal['현금','카드','기타']**) · `items`(**list[ReceiptItem]**)

<details><summary>힌트</summary>

```text
세부구현:
1. BaseModel 을 상속한 ReceiptItem 을 만든다(필드 3개).
2. Receipt 를 만들고 items 필드를 list[ReceiptItem] 으로 선언한다.
3. store_name 은 Optional[str] = Field(default=None, ...) 로 두고 description 에 '안 보이면 null' 을 적는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
fields = Receipt.model_fields
assert set(fields) == {'store_name', 'total_amount', 'payment', 'items'}
assert fields['store_name'].is_required() is False, 'store_name 은 Optional 이어야 합니다(안 보이면 None)'
assert set(ReceiptItem.model_fields) == {'name', 'quantity', 'price'}
print('✅ 통과!')

> 스키마를 직접 설계해 본 것으로 1단계는 끝입니다. **2단계부터는 아래 제공 셀의 스키마를 씁니다** — 뒤 단계의 채점 기준(품목 수·검산)이 이 스키마를 전제로 하기 때문입니다.

In [ ]:
# [제공 코드] 2단계부터 쓸 스키마 (이 셀을 실행하면 위에서 만든 것을 덮어씁니다)
from pydantic import BaseModel, Field
from typing import Literal, Optional

class ReceiptItem(BaseModel):
    name: str = Field(description='품목명. 영수증에 적힌 그대로')
    quantity: int = Field(description='수량. 안 적혀 있으면 1')
    price: float = Field(description='그 줄의 합계 금액')

class Receipt(BaseModel):
    store_name: Optional[str] = Field(default=None, description='가게 이름. 안 보이면 null')
    total_amount: float = Field(description='영수증에 적힌 총 결제 금액')
    payment: Literal['현금', '카드', '기타'] = Field(description='결제 수단')
    items: list[ReceiptItem] = Field(description='품목 목록')
print('제공 스키마로 진행합니다')

### 2단계 — 사진 3장 읽어 표 만들기
`data/receipt` 폴더의 사진을 이름순으로 정렬해 **앞 3장**을 읽으세요. 제공된 `to_data_url()` 을 씁니다.

- `parse`(`model='gpt-4o-mini'`, `response_format=Receipt`)로 각 사진을 분석하세요. 질문 텍스트는 **'이 영수증에서 가게·총액·결제수단·품목을 스키마대로 읽어줘. 안 보이는 값은 null.'** 로 하세요.
- 영수증 단위 정보를 `{'image_file': 파일명, 'store_name': ..., 'total_amount': ..., 'payment': ...}` 로 리스트 **`receipt_rows`** 에, 품목을 `{'image_file': 파일명, 'name': ..., 'price': ...}` 로 리스트 **`item_rows`** 에 담으세요.
- 각각 DataFrame **`receipt_df`**·**`item_df`** 로 만드세요.

<details><summary>힌트</summary>

```text
세부구현:
1. sorted(Path(...).glob('*.jpg'))[:3] 로 사진 3장을 고른다.
2. 사진마다 parse 를 호출하고 .parsed 를 꺼낸다.
3. 영수증 정보는 receipt_rows 에, r.items 를 돌며 품목은 item_rows 에 담는다(둘 다 image_file 을 함께).
4. pd.DataFrame 으로 각각 만든다.
```

</details>

In [ ]:
# [제공 코드] 이미지 → data URL 헬퍼 (교안에서 본 그대로)
import base64
from pathlib import Path

def to_data_url(path):
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'data:image/jpeg;base64,{b64}'

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(receipt_rows) == 3, '사진 3장을 모두 읽어야 합니다'
assert all({'image_file', 'store_name', 'total_amount', 'payment'} <= set(r.keys()) for r in receipt_rows), \
    '지문이 요구한 키가 다 있어야 합니다(item_count 같은 열을 더 담는 것은 괜찮습니다)'
assert len(item_rows) >= 3, '영수증 3장에서 품목이 최소 3줄은 나와야 합니다'
assert all({'image_file', 'name', 'price'} <= set(r.keys()) for r in item_rows), \
    '지문이 요구한 키가 다 있어야 합니다(quantity 같은 열을 더 담는 것은 괜찮습니다)'
assert item_df['image_file'].nunique() >= 2, '한 장만 처리하고 끝내지 않았는지 확인하세요'
print('✅ 통과!')

### 3단계 — 검산: 품목 합 == 총액
OCR 은 숫자를 잘못 읽습니다(0↔8, 1↔7). 영수증에는 검산 수단이 있습니다 — **품목 금액의 합이 총액과 같아야** 합니다.

- `item_df` 를 `image_file` 로 묶어 `price` 합을 구하고, `receipt_df` 의 `total_amount` 와 나란히 놓은 DataFrame **`verify_df`** 를 만드세요(열 이름은 자유).
- 두 값의 차이가 **1 미만이면 통과**로 보고, 통과한 영수증 수를 정수 **`ok_count`** 에 담으세요.
- 이어서 사람이 정리한 정답표(`receipts.csv`)의 `total_amount` 와도 대조해, **모델이 읽은 총액이 정답표와 맞는 장수**를 정수 **`match_count`** 에 담으세요(차이 1 미만이면 일치).

<details><summary>힌트</summary>

```text
세부구현:
1. item_df.groupby('image_file')['price'].sum() 으로 영수증별 품목 합을 구한다.
2. receipt_df 를 image_file 기준으로 그 합과 합친다(merge 또는 set_index+join).
3. (총액 - 품목합).abs() < 1 인 행 수를 세어 ok_count 에 담는다.
4. receipts.csv 를 읽어 image_file 로 merge 한 뒤, 총액 차이가 1 미만인 행 수를 match_count 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert verify_df.shape[0] == 3
assert isinstance(ok_count, int) and 0 <= ok_count <= 3
# 검산 기준이 실제로 계산됐는지 — 총액과 품목 합 두 값이 모두 표에 있어야 한다
assert verify_df.select_dtypes('number').shape[1] >= 2, '총액과 품목 합이 모두 들어 있어야 합니다'
# 금액이 전부 0 이면 영수증을 제대로 읽지 못한 것이다
assert float(verify_df.select_dtypes('number').abs().to_numpy().sum()) > 0, \
    '금액이 전부 0 입니다 — 2단계가 영수증을 제대로 읽었는지 확인하세요'
assert isinstance(match_count, int) and 0 <= match_count <= 3
assert set(['total_amount_모델', 'total_amount_정답']) <= set(cmp.columns), '정답표와 나란히 놓고 비교해야 합니다'
print('✅ 통과!  (검산 통과 수는 영수증에 따라 다릅니다 — 할인·부가세가 따로 적힌 영수증은 안 맞습니다)')

### 4단계 — 결과 저장
`receipt_df` 를 **`output/receipts_parsed.csv`** 로, `item_df` 를 **`output/receipt_items_parsed.csv`** 로 저장하세요(둘 다 `index=False`, `encoding='utf-8-sig'`). 저장한 영수증 표 경로를 **`parsed_path`** 에 담으세요.

<details><summary>힌트</summary>

```text
세부구현:
1. os.makedirs('output', exist_ok=True) 로 폴더를 준비한다.
2. to_csv 로 두 파일을 저장한다(index=False, encoding='utf-8-sig').
3. 영수증 표 경로 문자열을 parsed_path 에 담는다.
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
assert os.path.exists(parsed_path)
reloaded = pd.read_csv(parsed_path)
assert reloaded.shape[0] == 3, '영수증 3장이 그대로 저장돼야 합니다'
assert 'total_amount' in reloaded.columns
assert os.path.exists('output/receipt_items_parsed.csv'), '품목 표도 저장해야 합니다'
print('✅ 통과!')

---
수고했어요! 오늘 배운 대화·파라미터·프롬프트·Function Calling·구조화 출력·**이미지 정형화**를 묶어 **리포트 생성기**·**도구 쓰는 어시스턴트**·**영수증 정산기**를 완성했습니다. 이 경험이 다음 과목(RAG·에이전트)의 바탕이 됩니다.